# EA Stakeholder Mapping Analysis

This notebook demonstrates the complete workflow for analyzing EA conference attendee data:

1. **Data Loading**: Load and preprocess attendee data
2. **Extraction**: Extract locations, organizations, and cause areas using NLP and LLM methods
3. **Comparison**: Compare extraction methods (NLP vs LLM)
4. **Geographic Analysis**: Visualize geographic distribution
5. **Semantic Analysis**: Cluster and visualize cause areas by similarity

In [1]:
%reload_ext autoreload
%autoreload 2
# Setup
import sys

sys.path.insert(0, "..")

import pandas as pd

pd.set_option("display.max_colwidth", 100)

# Import the package
import sm
from sm import Pipeline, NLPExtractor, LLMExtractor, check_ollama_available

print(f"SM version: {sm.__version__}")

SM version: 0.3.0


## 1. Check Setup

Before running LLM extraction, verify Ollama is running:

In [2]:
# Check Ollama availability
available, msg = check_ollama_available()
print(f"Ollama: {msg}")

# Check cache status
stats = sm.get_cache_stats()
print(f"\nCache: {stats['total_files']} files, {stats['total_size_mb']} MB")

Ollama: Ollama available with model 'llama3.2'

Cache: 4188 files, 0.54 MB


## 2. Create Pipeline and Load Data

In [3]:
# Create pipeline with both extraction methods
# Change methods to ["nlp"] if Ollama is not available
pipe = Pipeline(
    methods=["nlp", "llm"],  # Use both methods for comparison
    llm_n_runs=3,  # Run LLM 3 times for majority voting
    use_cache=True,  # Cache results to avoid recomputation
)

# Load data (adjust path as needed)
df = pipe.load_data()  # Uses default path

# Or use sample data for testing
# from sm.data import get_sample_data
# df = pipe.set_data(df.head(300))

Loaded 650 rows with columns: ['company', 'job', 'career', 'biography', 'expertise', 'interests', 'help_me', 'help_others']


In [4]:
# Count number of mentions of 'ai' and 'climate' anywhere in the data (case-insensitive)
ai_counts = 0
climate_counts = 0

for col in df.columns:
    # Only consider columns with string/object dtype (skip numbers, etc)
    if df[col].dtype == object:
        # fillna('') to avoid errors with NaN cells
        for val in df[col].fillna(""):
            cell_str = str(val).lower()
            if "ai" in cell_str:
                ai_counts += 1
            if "climate" in cell_str:
                climate_counts += 1

print(f"AI mentions: {ai_counts}")
print(f"Climate mentions: {climate_counts}")


AI mentions: 1182
Climate mentions: 274


In [5]:
# Check what AI-related cause areas were extracted
all_causes = []
for col in ["llm_cause_areas", "nlp_cause_areas", "expertise_parsed", "interests_parsed"]:
    if col in df.columns:
        for items in df[col]:
            if isinstance(items, list):
                all_causes.extend([str(x).lower().strip() for x in items if x])

from collections import Counter

counts = Counter(all_causes)

# Find all AI-related
ai_related = {
    k: v
    for k, v in counts.items()
    if any(term in k for term in ["ai", "artificial intelligence", "alignment", "machine learning"])
}

print("AI-related extractions:")
for area, count in sorted(ai_related.items(), key=lambda x: x[1], reverse=True):
    print(f"  {count:4d}x: {area}")
print(f"\nTotal AI-related mentions: {sum(ai_related.values())}")

# Check climate
climate_related = {
    k: v
    for k, v in counts.items()
    if any(term in k for term in ["climate", "environment", "carbon"])
}
print("\nClimate-related extractions:")
for area, count in sorted(climate_related.items(), key=lambda x: x[1], reverse=True):
    print(f"  {count:4d}x: {area}")
print(f"\nTotal climate-related mentions: {sum(climate_related.values())}")


AI-related extractions:

Total AI-related mentions: 0

Climate-related extractions:

Total climate-related mentions: 0


## 3. Run Extraction

Extract locations, organizations, and cause areas from text columns:

In [6]:
# Run extraction on freeform text columns
df = pipe.extract(
    text_columns=["biography", "help_me", "help_others"],  # Freeform text columns
    semicolon_columns=["expertise", "interests"],  # Structured columns
    progress=True,
)

# View extracted columns
extract_cols = [c for c in df.columns if "nlp_" in c or "llm_" in c or "_parsed" in c]
print(f"\nExtracted columns: {extract_cols}")

NLP extraction:   0%|          | 0/650 [00:00<?, ?it/s]

LLM extraction (parallel):   0%|          | 0/650 [00:00<?, ?it/s]


Extracted columns: ['expertise_parsed', 'interests_parsed', 'nlp_locations', 'nlp_organizations', 'nlp_cause_areas', 'llm_locations', 'llm_organizations', 'llm_cause_areas']


In [7]:
num_preview = 3
# view original text
print("Original text:")
for i, row in df.head(num_preview).iterrows():
    print(
        "  Row "
        + str(i)
        + ": "
        + row["biography"]
        + "\t"
        + str(row["expertise_parsed"])
        + "\t"
        + str(row["interests_parsed"])
    )

# View sample extractions
if "nlp_cause_areas" in df.columns:
    print("\nNLP Cause Areas (sample):")
    for i, areas in enumerate(df["nlp_cause_areas"].head(num_preview)):
        print(f"  Row {i}: {areas}")

if "llm_cause_areas" in df.columns:
    print("\nLLM Cause Areas (sample):")
    for i, areas in enumerate(df["llm_cause_areas"].head(num_preview)):
        print(f"  Row {i}: {areas}")

Original text:
  Row 0: I am Ghaniyyah Abdulkareem, a passionate advocate for social impact, climate justice, and youth empowerment. With a degree in English from Bayero University, Kano, I have actively served in various leadership roles including Treasurer and General Secretary of JCI Nigeria (BUK), Assistant General Secretary of the National Association of English and Literary Studies, and General Secretary of the Yoruba Student Union. I also coordinate initiatives for Safe Migration and Social Justice in Kwara State and serve as the Kano State Coordinator for English and Literary Studies students.

My work focuses on Migration, Climate Action, Education, and Food Security. I have led and participated in numerous community development projects—such as organising clean-up campaigns, climate awareness drives, and feeding initiatives for Almajiri households. I'm also a Millennium Fellow and a trained advocate in Sustainable Development Goals, Climate Finance, and Grant Writing. With de

## 4. Compare Extraction Methods

Compare NLP vs LLM extraction performance:

In [8]:
# Compare methods (only if both are available)
# TODO: check that uniquified
if pipe.nlp and pipe.llm:
    comparison = pipe.compare_methods(text_columns=["biography"])
    print(comparison.summary())

Comparing extractors:   0%|          | 0/650 [00:00<?, ?it/s]

EXTRACTION METHOD COMPARISON
Texts analyzed: 488

LOCATIONS:
  NLP: 470 total (1.0/text)
  LLM: 1353 total (2.8/text)
  Overlap: 311

ORGANIZATIONS:
  NLP: 1187 total (2.4/text)
  LLM: 793 total (1.6/text)
  Overlap: 320

CAUSE AREAS:
  NLP: 3217 total (6.6/text)
  LLM: 2354 total (4.8/text)
  Overlap: 246

OVERALL:
  Total NLP extractions: 4874
  Total LLM extractions: 4500
  Total overlap: 877
  Overall Jaccard similarity: 10.32%


In [9]:
# Visualize comparison
if pipe.results.comparison:
    from sm.viz import create_extraction_comparison_chart

    fig = create_extraction_comparison_chart(comparison.to_dataframe(), showtitle=False)
    fig.show()
    fig.write_html(
        "output/extraction_comparison_chart.html",
        include_plotlyjs="cdn",
        full_html=True,
        config={"responsive": True},
    )

## 5. Geographic Analysis

Analyze geographic distribution of attendees:

In [10]:
# Run geographic analysis (requires API access for geocoding)

# TODO: improve organisation filtering
try:
    country_df, org_df = pipe.analyze_geographic(force_reload=False)
    print("\nCountry mentions:")
    display(country_df.head(10))
except Exception as e:
    print(f"Geographic analysis skipped: {e}")

Geocoding locations:   0%|          | 0/575 [00:00<?, ?it/s]

Geocoding organizations:   0%|          | 0/829 [00:00<?, ?it/s]


Country mentions:


,country,count,attendee_ids
0,United States,153,"[1, 6, 14, 15, 18, 34, 36, 38, 39, 46, 49, 50, 56, 60, 61, 70, 79, 85, 93, 104, 111, 117, 118, 1..."
1,Malta,138,"[1, 12, 14, 18, 34, 36, 38, 39, 46, 49, 50, 56, 57, 70, 79, 85, 93, 104, 111, 117, 119, 123, 124..."
2,The Netherlands,120,"[9, 15, 18, 20, 21, 36, 43, 46, 49, 50, 54, 56, 60, 61, 72, 79, 82, 84, 95, 96, 98, 102, 105, 11..."
3,United Kingdom,71,"[1, 34, 36, 54, 59, 70, 84, 85, 93, 136, 153, 158, 164, 165, 172, 179, 185, 198, 211, 218, 221, ..."
4,France,44,"[35, 49, 57, 61, 77, 79, 129, 138, 144, 149, 168, 170, 174, 177, 187, 206, 226, 234, 245, 309, 3..."
5,Mexico,15,"[20, 134, 149, 162, 207, 209, 327, 363, 411, 421, 504, 573, 610, 632, 633]"
6,Cuba,12,"[54, 180, 205, 299, 316, 346, 385, 451, 504, 505, 521, 610]"
7,India,9,"[103, 104, 245, 269, 376, 429, 464, 480, 612]"
8,Germany,8,"[221, 224, 457, 480, 525, 526, 626, 644]"
9,Switzerland,7,"[141, 170, 241, 266, 390, 406, 525]"


In [11]:
# Create map visualization
if pipe.results.country_counts is not None:
    fig = pipe.create_map(show_organizations=True, showtitle=False)
    fig.show()
    fig.write_html(
        "output/map.html",
        include_plotlyjs="cdn",
        full_html=True,
        config={"responsive": True},
    )

## 6. Semantic Analysis

Cluster cause areas by semantic similarity:

In [13]:
# Run semantic analysis
try:
    semantic_result = pipe.analyze_semantic(
        semicolon_columns=["expertise", "interests"],
        min_mentions=1,  # Lower threshold for sample data
        min_category_size=5,
        use_predefined_categories=True,
        normalize_terms=True,
    )

    cluster_str = "clusters" if len(semantic_result.clusters) > 1 else "cluster"
    # print(f"Automatically organised these into {len(semantic_result.clusters)} {cluster_str}")

    # Show cluster summary
    print("\nCluster Summary:")
    display(semantic_result.get_cluster_summary())

except ImportError as e:
    print(f"Semantic analysis requires additional packages: {e}")
    print("Install with: pip install sentence-transformers scikit-learn")
except Exception as e:
    print(f"Semantic analysis error: {e}")

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: d6e61370-7301-4a67-a2ea-1a06abd8e1c8)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].


Found 194 unique cause areas (minimum 1 mention)
  (with term normalization applied - similar terms grouped together)


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json (Caused by NameResolutionError("HTTPSConnection(host=\'huggingface.co\', port=443): Failed to resolve \'huggingface.co\' ([Errno 8] nodename nor servname provided, or not known)"))'), '(Request ID: 4ba362e4-6c7b-4b50-a277-2e02641bb6f7)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/./modules.json
Retrying in 2s [Retry 2/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 9e96391b-0d3f-4a0d-ba50-692698a27241)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json
Retrying in 1s [Retry 1/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max ret

Automatically organised these into 12 clusters using predefined categories

Cluster Summary:


,cluster_id,name,n_members,total_mentions,top_members
0,0,Operations & Management,26,6397,"community management, operations, software engineering"
1,1,Research & Academia,18,4225,"academic research, data science, entrepreneurship"
2,2,AI Safety & Governance,11,1917,"ai strategy & policy, ai safety technical research, ai alignment"
3,3,Animal Welfare,12,1731,"farmed animal welfare, alternative proteins, biosecurity"
4,4,Global Health,14,1697,"global mental health & well-being, healthcare, medicine"
5,5,Policy & Governance,22,1501,"policymaking, improving institutional decision making, politics"
6,6,Other,43,1255,"machine learning, earning to give, user experience design"
7,7,Global Poverty & Development,11,1107,"global priorities research, global coordination & peace-building, civilisational recovery"
8,8,Climate & Environment,13,714,"climate change mitigation, environmental policy, sustainability"
9,9,EA Community & Meta,6,681,"ea community building, metaethics, discussion forum design"


In [ ]:
# Create cause area bar chart
if pipe.results.cause_areas:
    fig = pipe.create_cause_area_chart(top_n=20, showtitle=False)
    fig.show()
    fig.write_html(
        "output/cause_area_chart.html",
        include_plotlyjs="cdn",
        full_html=True,
        config={"responsive": True},
    )

In [ ]:
# Create semantic network visualization
if pipe.results.semantic_result:
    fig = pipe.create_semantic_network(min_mentions=3, showtitle=False)
    fig.show()
    fig.write_html(
        "output/semantic_network.html",
        include_plotlyjs="cdn",
        full_html=True,
        config={"responsive": True},
    )

In [15]:
# Visualise ratio of expertise to interest
fig = pipe.create_expertise_vs_interest_chart(showtitle=False)
fig.show()
fig.write_html(
    "output/expertise_vs_interest_chart.html",
    include_plotlyjs="cdn",
    full_html=True,
    config={"responsive": True},
)

In [16]:
# Determine under-valued areas (where interest exceeds expertise)
fig = pipe.create_undervalued_chart(top_n=30, showtitle=False)
fig.show()
fig.write_html(
    "output/undervalued_chart.html",
    include_plotlyjs="cdn",
    full_html=True,
    config={"responsive": True},
)

## 7. Additional Visualizations

In [ ]:
# Cluster treemap
if pipe.results.semantic_result:
    from sm.viz import create_cluster_treemap

    fig = create_cluster_treemap(pipe.results.semantic_result)
    fig.show()
    fig.write_html(
        "output/cluster_treemap.html",
        include_plotlyjs="cdn",
        full_html=True,
        config={"responsive": True},
    )

In [ ]:
# Similarity heatmap
if pipe.results.semantic_result:
    from sm.viz import create_similarity_heatmap

    fig = create_similarity_heatmap(pipe.results.semantic_result, top_n=15)
    fig.show()
    fig.write_html(
        "output/similarity_heatmap.html",
        include_plotlyjs="cdn",
        full_html=True,
        config={"responsive": True},
    )

## 8. Personal Recommendations

In [ ]:
# Create recommender from pipeline
recommender = pipe.create_recommender(augment_with_extraction="llm")

# Get recommendations for person at index 0
recs = recommender.recommend(person_idx=552, top_k=3)

print(recs.summary())

Computing person embeddings...
  Augmenting with LLM cause areas
  Found 122 people with empty profiles (will be excluded)


Batches:   0%|          | 0/21 [00:00<?, ?it/s]

Computed embeddings for 528 people (122 excluded due to empty profiles)
Recommendations for '552'

🤝 Similar (collaboration – high profile similarity):
  • Person 538 (73%)
      Shared interests: machine learning, software development, climate change mitigation
  • Person 276 (71%)
      Shared interests: 
  • Person 186 (70%)
      Shared interests: climate change mitigation, people management, academic research

🔄 Complementary (cross-pollination – moderate similarity, different focus areas):
  • Person 641 (48%)
      Common ground: academic research, machine learning safety. They bring: policymaking, factory farming
  • Person 310 (47%)
      Common ground: data visualization, machine learning. They bring: supply chain transparency, ai strategy & policy
  • Person 293 (47%)
      Common ground: communications, people management. They bring: factory farming, hr

🎯 Skill Match (expertise ↔ interests):
  • Person 2 (81%)
      They have expertise in: data visualization, machine learn

## 9. Cache Management

In [ ]:
# View cache statistics
stats = sm.get_cache_stats()
print(f"Cache Statistics:")
print(f"  Total files: {stats['total_files']}")
print(f"  Total size: {stats['total_size_mb']} MB")
print(f"  Categories: {list(stats['categories'].keys())}")

Cache Statistics:
  Total files: 4188
  Total size: 0.54 MB
  Categories: ['organizations', 'geonames', 'llm', 'geonames_orgs', 'geocoding', 'locations', 'nlp', 'keywords']


In [ ]:
# Clear cache if needed (uncomment to run)
# deleted = sm.clear_cache()  # Clear all
# deleted = sm.clear_cache("llm")  # Clear only LLM cache
# print(f"Deleted {deleted} cache files")